In [1]:
%pip install langchain langchain-community langchain-ollama langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import cv2
import json
import whisper
import yt_dlp
import numpy as np
from PIL import Image
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=TXChIdPaJ3g"
OUTPUT_DIR = "video_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download video (for frames) and audio (for transcript)
ydl_opts = {
    "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4",
    "outtmpl": f"{OUTPUT_DIR}/video.%(ext)s",
    "merge_output_format": "mp4",
    "ffmpeg_location": "/usr/local/bin/ffmpeg",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([VIDEO_URL])

VIDEO_PATH = f"{OUTPUT_DIR}/video.mp4"
print("Download complete")

[youtube] Extracting URL: https://www.youtube.com/watch?v=TXChIdPaJ3g
[youtube] TXChIdPaJ3g: Downloading webpage


[youtube] TXChIdPaJ3g: Downloading android vr player API JSON
[info] TXChIdPaJ3g: Downloading 1 format(s): 399+140
[download] Destination: video_data/video.f399.mp4
[download] 100% of   58.52MiB in 00:00:07 at 7.47MiB/s     
[download] Destination: video_data/video.f140.m4a
[download] 100% of   23.81MiB in 00:00:02 at 8.03MiB/s     
[Merger] Merging formats into "video_data/video.mp4"
Deleting original file video_data/video.f399.mp4 (pass -k to keep)
Deleting original file video_data/video.f140.m4a (pass -k to keep)
✅ Download complete


In [ ]:
# Load Whisper model (use "base" for speed, "medium"/"large" for accuracy)
model = whisper.load_model("base")

result = model.transcribe(VIDEO_PATH, verbose=False)

# Save transcript chunks with timestamps
transcript_chunks = []
for seg in result["segments"]:
    transcript_chunks.append({
        "content": seg["text"].strip(),
        "timestamp": f"{int(seg['start']//60):02d}:{int(seg['start']%60):02d}",
        "start_sec": seg["start"],
        "modality": "text"
    })

print(f"Transcribed {len(transcript_chunks)} segments")
print(transcript_chunks[0])  # preview

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detected language: English


100%|█████████▉| 153900/154257 [01:32<00:00, 1665.01frames/s]

✅ Transcribed 386 segments
{'content': 'English Leap Podcast', 'timestamp': '00:00', 'start_sec': 0.0, 'modality': 'text'}


In [ ]:
FRAME_INTERVAL_SEC = 30  # extract one frame every 30 seconds

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * FRAME_INTERVAL_SEC)

frames_dir = f"{OUTPUT_DIR}/frames"
os.makedirs(frames_dir, exist_ok=True)

frame_paths = []
frame_idx = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        timestamp_sec = frame_idx / fps
        timestamp_str = f"{int(timestamp_sec//60):02d}:{int(timestamp_sec%60):02d}"
        path = f"{frames_dir}/frame_{timestamp_str.replace(':','_')}.jpg"
        cv2.imwrite(path, frame)
        frame_paths.append({"path": path, "timestamp": timestamp_str, "timestamp_sec": timestamp_sec})
        saved += 1
    frame_idx += 1

cap.release()
print(f"Saved {saved} frames")

✅ Saved 52 frames


In [ ]:
import base64

llava = OllamaLLM(model="llava")

def describe_frame(image_path):
    """Send a frame to LLaVA for visual description."""
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")
    
    # LLaVA via langchain-ollama supports images this way
    from langchain_ollama import ChatOllama
    from langchain_core.messages import HumanMessage

    
    chat = ChatOllama(model="llava")
    msg = HumanMessage(content=[
        {"type": "text", "text": "Describe what is shown in this video frame in 2-3 sentences. Focus on key visual elements, text on screen, diagrams, or actions."},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_data}"}
    ])
    response = chat.invoke([msg])
    return response.content

# Describe all frames (this may take a few minutes)
image_chunks = []
for frame_info in frame_paths:
    print(f"Describing {frame_info['timestamp']}...", end=" ")
    description = describe_frame(frame_info["path"])
    image_chunks.append({
        "content": description,
        "timestamp": frame_info["timestamp"],
        "modality": "image"
    })
    print("✅")

print(f"\nDone! Described {len(image_chunks)} frames")

Describing 00:00... ✅
Describing 00:30... ✅
Describing 01:00... ✅
Describing 01:30... ✅
Describing 02:00... ✅
Describing 02:30... ✅
Describing 03:00... ✅
Describing 03:30... ✅
Describing 04:00... ✅
Describing 04:30... ✅
Describing 05:00... ✅
Describing 05:30... ✅
Describing 06:00... ✅
Describing 06:30... ✅
Describing 07:00... ✅
Describing 07:30... ✅
Describing 08:00... ✅
Describing 08:30... ✅
Describing 09:00... ✅
Describing 09:30... ✅
Describing 10:00... ✅
Describing 10:30... ✅
Describing 11:00... ✅
Describing 11:30... ✅
Describing 12:00... ✅
Describing 12:30... ✅
Describing 13:00... ✅
Describing 13:30... ✅
Describing 14:00... ✅
Describing 14:30... ✅
Describing 15:00... ✅
Describing 15:30... ✅
Describing 16:00... ✅
Describing 16:30... ✅
Describing 17:00... ✅
Describing 17:30... ✅
Describing 18:00... ✅
Describing 18:30... ✅
Describing 19:00... ✅
Describing 19:30... ✅
Describing 20:00... ✅
Describing 20:30... ✅
Describing 21:00... ✅
Describing 21:30... ✅
Describing 22:00... ✅
Describing

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Combine text + image chunks into LangChain Documents
all_chunks = transcript_chunks + image_chunks

documents = [
    Document(
        page_content=chunk["content"],
        metadata={
            "timestamp": chunk["timestamp"],
            "modality": chunk["modality"]
        }
    )
    for chunk in all_chunks
]

# Store in ChromaDB (persisted to disk)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Stored {len(documents)} chunks in ChromaDB")

✅ Stored 438 chunks in ChromaDB
